In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
import os
from pandas.api.types import CategoricalDtype


# Main tables

### Table 1 - Gaia DR3 parameters
This table should have the following:
1. star ID
2. Gaia DR3 ID
3. RA
4. Dec
5. $l$
6. $b$
7. $\pi$
8. $\pi$ error
9. m$_{G}$
10. E(B-V)
11. Orbit

In [6]:
# Loading in the dataset
prop_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/stellar_params/emp_params_FINAL.csv')
table1 = prop_df[['star_id', 'gaia_id', 'ra', 'dec', 'l', 'b', 'parallax', 'parallax_error', 'gmag', 'ebv_corr', 'orbit']]

table1 = table1.rename(columns={'parallax': 'plx'})
table1.sort_values(by='star_id', inplace=True)
display(table1)

# Storing into .fits file
t1_fits = Table.from_pandas(table1)
hdu1_fits = fits.BinTableHDU(t1_fits, name='GAIA_PARAMS')
hdul = fits.HDUList([fits.PrimaryHDU(), hdu1_fits])

hdul.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_1.fits', overwrite=True)
table1.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_1.csv', index=False)

,star_id,gaia_id,ra,dec,l,b,plx,parallax_error,gmag,ebv_corr,orbit
0,ra_0103-7050_s163,4689369645972422784,13.400967,-71.079819,302.678953,-46.047367,0.736594,0.025971,15.560211,0.039814,pro_disk
1,ra_0834-5220_s316,5321186578181666176,129.069598,-52.943230,269.903058,-7.297586,0.646204,0.045634,16.789020,0.507827,pro_disk
2,ra_1604-2712_s24,6043161513972081024,240.420121,-26.680890,347.281224,19.382796,-0.016893,0.047017,15.533642,0.103916,halo
3,ra_1604-2712_s292,6042817710432143104,240.530939,-27.701053,346.606149,18.580956,0.186226,0.063081,16.774326,0.123317,halo
4,ra_1624-2150_s278,6052240868671385472,246.051779,-21.324422,355.150931,19.323151,0.291232,0.084932,17.255495,0.428400,halo
5,ra_1633-2814_s130,6044035900595959424,248.705020,-28.469377,351.181408,12.819284,0.196550,0.084237,16.882343,0.502348,gse
6,ra_1633-2814_s284,6044482989511923584,248.310989,-28.051426,351.271192,13.355463,0.248548,0.101894,17.126230,0.404852,gse
7,ra_1648-0653_s38,4340844491685350912,251.316703,-6.678217,11.023506,24.166968,0.247956,0.075203,16.966896,0.409112,pro_disk
8,ra_1656-1433_s143,4140336967130944640,254.055528,-14.113023,6.088882,17.738968,0.052396,0.034812,15.423788,0.669225,halo
9,ra_1658-2454_s22,4113281490693917568,253.681095,-24.489046,357.218894,11.879206,0.134216,0.033764,15.336399,0.319906,halo


### Table 2 - Observation log
This should have the following:
1. star ID
2. N exposures
3. SN uvb
4. SN vis
5. RV helio
6. RV helio error

In [89]:
# Loading the data with exposure times, number of OBs and dates of observations
exposures_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/stellar_params/emp_obs_dates.csv')
# exposures_df = pd.read_csv('/data/misfit/benlowe/xshooter/p115/stellar_params/emp_obs_dates.csv')


# Loading in the SNR and RVs data
snr_rv_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/reduced_data/comb_spec_3/star_info.csv')
# snr_rv_df = pd.read_csv('/data/misfit/benlowe/xshooter/p115/reduced_data/comb_spec/star_info.csv')
print(snr_rv_df.columns)

# Merging dataframes together, then applying the heliocentric correction to the RVs
table_data = pd.merge(exposures_df, snr_rv_df[['star_id', 'snr_uvb', 'snr_vis', 'rv_avg', 'rv_avg_err', 'rv_uvb', 'rv_uvb_err', 'rv_vis', 'rv_vis_err', 'heliocor', 'barycor']], on='star_id', how='left')
table_data['rv_uvb_helio'] = table_data['rv_uvb'] + table_data['heliocor']
table_data['rv_vis_helio'] = table_data['rv_vis'] + table_data['heliocor']
table_data['rv_avg_helio'] = table_data['rv_avg'] + table_data['heliocor']

table_data['rv_uvb_bary'] = table_data['rv_uvb'] + table_data['barycor']
table_data['rv_vis_bary'] = table_data['rv_vis'] + table_data['barycor']
table_data['rv_avg_bary'] = table_data['rv_avg'] + table_data['barycor']
# table_data['rv_uvb_bary']


table_data = table_data[['star_id', 'obs', 'exps', 'snr_uvb', 'snr_vis', 'rv_avg', 'rv_avg_err']]
display(table_data)

# Storing into .fits file
t2_fits = Table.from_pandas(table_data)
hdu2_fits = fits.BinTableHDU(t2_fits, name='OBS_LOG')
hdul2 = fits.HDUList([fits.PrimaryHDU(), hdu2_fits])

hdul2.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_2.fits', overwrite=True)
table_data.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_2.csv', index=False)

Index(['star_id', 'snr_uvb', 'snr_vis', 'ra', 'dec', 'barycor', 'heliocor',
       'rv_avg', 'rv_avg_err', 'rv_uvb', 'rv_uvb_err', 'rv_vis', 'rv_vis_err',
       'rv_hdelta', 'rv_hdelta_err', 'rv_hgamma', 'rv_hgamma_err', 'rv_hbeta',
       'rv_hbeta_err', 'rv_halpha', 'rv_halpha_err', 'rv_caii849',
       'rv_caii849_err', 'rv_caii854', 'rv_caii854_err', 'rv_caii866',
       'rv_caii866_err'],
      dtype='object')


,star_id,obs,exps,snr_uvb,snr_vis,rv_avg,rv_avg_err
0,ra_0103-7050_s163,1,1800,125.759183,129.589052,-28.356684,0.210276
1,ra_0834-5220_s316,2,2760,88.695864,145.256296,143.008362,0.221615
2,ra_1604-2712_s24,1,2700,152.512449,207.383213,190.187346,0.147964
3,ra_1604-2712_s292,2,2100,84.429111,95.302581,-4.812705,0.308310
4,ra_1624-2150_s278,3,2520,108.125523,149.699693,249.527089,0.311230
5,ra_1633-2814_s130,4,3120,114.537858,237.310087,193.015129,0.149548
6,ra_1633-2814_s284,4,3120,122.487576,209.063615,178.335222,0.199538
7,ra_1648-0653_s38,3,3120,126.621239,184.752043,-12.851709,0.186835
8,ra_1656-1433_s143,2,3120,145.015302,347.470042,-53.080993,0.090079
9,ra_1658-2454_s22,1,2820,130.657811,227.897299,56.475531,0.096629


### Table 3 - Median errors
This table should have the following:
1. Element
2. Teff median error
3. Logg median error
4. Vmic median error

In [91]:
# Loading in the abundance data
path = '/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/'
abunds = [path + f for f in os.listdir(path) if f.endswith('I_abunds.csv')]

# Extracting the median errors for each element
al_I_abunds = pd.read_csv(abunds[0])
al_I_med_tefferr = al_I_abunds['bestfit_abund_tefferr_xfe'].median()
al_I_med_loggerr = al_I_abunds['bestfit_abund_loggerr_xfe'].median()
al_I_med_vmicerr = al_I_abunds['bestfit_abund_vmicerr_xfe'].median()

ba_II_abunds = pd.read_csv(abunds[1])
ba_II_med_tefferr = ba_II_abunds['bestfit_abund_tefferr_xfe'].median()
ba_II_med_loggerr = ba_II_abunds['bestfit_abund_loggerr_xfe'].median()
ba_II_med_vmicerr = ba_II_abunds['bestfit_abund_vmicerr_xfe'].median()

c_I_abunds = pd.read_csv(abunds[2])
c_I_med_tefferr = c_I_abunds['bestfit_abund_tefferr_xfe'].median()
c_I_med_loggerr = c_I_abunds['bestfit_abund_loggerr_xfe'].median()
c_I_med_vmicerr = c_I_abunds['bestfit_abund_vmicerr_xfe'].median()

ca_II_abunds = pd.read_csv(abunds[3])
ca_II_med_tefferr = ca_II_abunds['bestfit_abund_tefferr_xfe'].median()
ca_II_med_loggerr = ca_II_abunds['bestfit_abund_loggerr_xfe'].median()
ca_II_med_vmicerr = ca_II_abunds['bestfit_abund_vmicerr_xfe'].median()

co_I_abunds = pd.read_csv(abunds[4])
co_I_med_tefferr = co_I_abunds['bestfit_abund_tefferr_xfe'].median()
co_I_med_loggerr = co_I_abunds['bestfit_abund_loggerr_xfe'].median() 
co_I_med_vmicerr = co_I_abunds['bestfit_abund_vmicerr_xfe'].median()

cr_I_abunds = pd.read_csv(abunds[5])
cr_I_med_tefferr = cr_I_abunds['bestfit_abund_tefferr_xfe'].median()
cr_I_med_loggerr = cr_I_abunds['bestfit_abund_loggerr_xfe'].median()
cr_I_med_vmicerr = cr_I_abunds['bestfit_abund_vmicerr_xfe'].median()

eu_II_abunds = pd.read_csv(abunds[6])
eu_II_med_tefferr = eu_II_abunds['bestfit_abund_tefferr_xfe'].median()
eu_II_med_loggerr = eu_II_abunds['bestfit_abund_loggerr_xfe'].median()
eu_II_med_vmicerr = eu_II_abunds['bestfit_abund_vmicerr_xfe'].median()

mg_I_abunds = pd.read_csv(abunds[7])
mg_I_med_tefferr = mg_I_abunds['bestfit_abund_tefferr_xfe'].median()
mg_I_med_loggerr = mg_I_abunds['bestfit_abund_loggerr_xfe'].median()
mg_I_med_vmicerr = mg_I_abunds['bestfit_abund_vmicerr_xfe'].median()

mn_I_abunds = pd.read_csv(abunds[8])
mn_I_med_tefferr = mn_I_abunds['bestfit_abund_tefferr_xfe'].median()
mn_I_med_loggerr = mn_I_abunds['bestfit_abund_loggerr_xfe'].median()
mn_I_med_vmicerr = mn_I_abunds['bestfit_abund_vmicerr_xfe'].median()

n_I_abunds = pd.read_csv(abunds[9])
n_I_med_tefferr = n_I_abunds['bestfit_abund_tefferr_xfe'].median()
n_I_med_loggerr = n_I_abunds['bestfit_abund_loggerr_xfe'].median()
n_I_med_vmicerr = n_I_abunds['bestfit_abund_vmicerr_xfe'].median()

na_I_abunds = pd.read_csv(abunds[10])
na_I_med_tefferr = na_I_abunds['bestfit_abund_tefferr_xfe'].median()
na_I_med_loggerr = na_I_abunds['bestfit_abund_loggerr_xfe'].median()
na_I_med_vmicerr = na_I_abunds['bestfit_abund_vmicerr_xfe'].median()

ni_I_abunds = pd.read_csv(abunds[11])
ni_I_med_tefferr = ni_I_abunds['bestfit_abund_tefferr_xfe'].median()
ni_I_med_loggerr = ni_I_abunds['bestfit_abund_loggerr_xfe'].median()
ni_I_med_vmicerr = ni_I_abunds['bestfit_abund_vmicerr_xfe'].median()

sc_II_abunds = pd.read_csv(abunds[12])
sc_II_med_tefferr = sc_II_abunds['bestfit_abund_tefferr_xfe'].median()
sc_II_med_loggerr = sc_II_abunds['bestfit_abund_loggerr_xfe'].median()
sc_II_med_vmicerr = sc_II_abunds['bestfit_abund_vmicerr_xfe'].median()

si_I_abunds = pd.read_csv(abunds[13])
si_I_med_tefferr = si_I_abunds['bestfit_abund_tefferr_xfe'].median()
si_I_med_loggerr = si_I_abunds['bestfit_abund_loggerr_xfe'].median()
si_I_med_vmicerr = si_I_abunds['bestfit_abund_vmicerr_xfe'].median()

sr_II_abunds = pd.read_csv(abunds[14])
sr_II_med_tefferr = sr_II_abunds['bestfit_abund_tefferr_xfe'].median()
sr_II_med_loggerr = sr_II_abunds['bestfit_abund_loggerr_xfe'].median()
sr_II_med_vmicerr = sr_II_abunds['bestfit_abund_vmicerr_xfe'].median()

ti_II_abunds = pd.read_csv(abunds[15])
ti_II_med_tefferr = ti_II_abunds['bestfit_abund_tefferr_xfe'].median()
ti_II_med_loggerr = ti_II_abunds['bestfit_abund_loggerr_xfe'].median()
ti_II_med_vmicerr = ti_II_abunds['bestfit_abund_vmicerr_xfe'].median()

fe_I_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/stellar_params.csv')
fe_I_med_tefferr = fe_I_abunds['Fe_tefferr'].median()
fe_I_med_loggerr = fe_I_abunds['Fe_loggerr'].median()
fe_I_med_vmicerr = fe_I_abunds['Fe_vmicerr'].median()


# # Storing into dataframe
# abund_data = {
#     'Element': ['Al I', 'Ba II', 'C I', 'Ca II', 'Co I', 'Cr I', 'Eu II', 'Mg I', 'Mn I', 'N I', 'Na I', 'Ni I', 'Sc II', 'Si I', 'Sr II', 'Ti II', 'Fe I'],
#     'Median Teff Error': [al_I_med_tefferr, ba_II_med_tefferr, c_I_med_tefferr, ca_II_med_tefferr, co_I_med_tefferr, cr_I_med_tefferr, eu_II_med_tefferr, mg_I_med_tefferr,
#                           mn_I_med_tefferr, n_I_med_tefferr, na_I_med_tefferr, ni_I_med_tefferr, sc_II_med_tefferr, si_I_med_tefferr, sr_II_med_tefferr, ti_II_med_tefferr, fe_I_med_tefferr],
#     'Median Logg Error': [al_I_med_loggerr, ba_II_med_loggerr, c_I_med_loggerr, ca_II_med_loggerr, co_I_med_loggerr, cr_I_med_loggerr,
#                           eu_II_med_loggerr, mg_I_med_loggerr, mn_I_med_loggerr, n_I_med_loggerr, na_I_med_loggerr, ni_I_med_loggerr,
#                           sc_II_med_loggerr, si_I_med_loggerr, sr_II_med_loggerr, ti_II_med_loggerr, fe_I_med_loggerr],
#     'Median Vmic Error': [al_I_med_vmicerr, ba_II_med_vmicerr, c_I_med_vmicerr, ca_II_med_vmicerr, co_I_med_vmicerr, cr_I_med_vmicerr,
#                           eu_II_med_vmicerr, mg_I_med_vmicerr, mn_I_med_vmicerr, n_I_med_vmicerr, na_I_med_vmicerr, ni_I_med_vmicerr,
#                           sc_II_med_vmicerr, si_I_med_vmicerr, sr_II_med_vmicerr, ti_II_med_vmicerr, fe_I_med_vmicerr]
# }
# Storing into dataframe (Sorted by Atomic Number)
abund_data = {
    'el': [
        'CH', 'NH', 'Na_I', 'Mg_I', 'Al_I', 'Si_I', 'Ca_II', 'Sc_II', 
        'Ti_II', 'Cr_I', 'Mn_I', 'Fe_I', 'Co_I', 'Ni_I', 'Sr_II', 'Ba_II', 'Eu_II'
    ],
    'teff_err': [
        c_I_med_tefferr, n_I_med_tefferr, na_I_med_tefferr, mg_I_med_tefferr, 
        al_I_med_tefferr, si_I_med_tefferr, ca_II_med_tefferr, sc_II_med_tefferr, 
        ti_II_med_tefferr, cr_I_med_tefferr, mn_I_med_tefferr, fe_I_med_tefferr, 
        co_I_med_tefferr, ni_I_med_tefferr, sr_II_med_tefferr, ba_II_med_tefferr, 
        eu_II_med_tefferr
    ],
    'logg_err': [
        c_I_med_loggerr, n_I_med_loggerr, na_I_med_loggerr, mg_I_med_loggerr, 
        al_I_med_loggerr, si_I_med_loggerr, ca_II_med_loggerr, sc_II_med_loggerr, 
        ti_II_med_loggerr, cr_I_med_loggerr, mn_I_med_loggerr, fe_I_med_loggerr, 
        co_I_med_loggerr, ni_I_med_loggerr, sr_II_med_loggerr, ba_II_med_loggerr, 
        eu_II_med_loggerr
    ],
    'vmic_err': [
        c_I_med_vmicerr, n_I_med_vmicerr, na_I_med_vmicerr, mg_I_med_vmicerr, 
        al_I_med_vmicerr, si_I_med_vmicerr, ca_II_med_vmicerr, sc_II_med_vmicerr, 
        ti_II_med_vmicerr, cr_I_med_vmicerr, mn_I_med_vmicerr, fe_I_med_vmicerr, 
        co_I_med_vmicerr, ni_I_med_vmicerr, sr_II_med_vmicerr, ba_II_med_vmicerr, 
        eu_II_med_vmicerr
    ]
}


abunds_df = pd.DataFrame(abund_data)
display(abunds_df)

# Storing into .fits file
t3_fits = Table.from_pandas(abunds_df)
hdu3_fits = fits.BinTableHDU(t3_fits, name='MED_ERRORS')
hdul3 = fits.HDUList([fits.PrimaryHDU(), hdu3_fits])

hdul3.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_3.fits', overwrite=True)
abunds_df.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_3.csv', index=False)

,el,teff_err,logg_err,vmic_err
0,CH,0.225463,0.110722,0.004166
1,NH,0.243363,0.116289,0.029593
2,Na_I,0.122520,0.042947,0.110079
3,Mg_I,0.096384,0.072158,0.088811
4,Al_I,0.071807,0.010539,0.103535
5,Si_I,0.148095,0.118677,0.097683
6,Ca_II,0.062893,0.013576,0.017714
7,Sc_II,0.042552,0.138333,0.065780
8,Ti_II,0.039119,0.090142,0.068654
9,Cr_I,0.119769,0.004889,0.041042


### Table 4 - Stellar parameters
This table should have the following:
1. star ID
2. Teff
3. Teff err
4. Logg
5. Logg err
6. [Fe/H]
7. [Fe/H] err
8. vmic
9. vmic err


In [22]:
# Loading in the stellar parameters
params_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/stellar_params.csv')
# display(params_df)
# print(params_df.keys())

# Loading in the radial velocities and SNRs
rv_snr_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/reduced_data/comb_spec_2/star_info.csv')
# display(rv_snr_df)
# print(rv_snr_df.keys())

# Loading in the teffs and loggs from the AAT data
aat_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/stellar_params/emp_params_FINAL.csv')
# display(aat_df)
# print(aat_df.keys())

# Merging the datasets
table4 = pd.merge(params_df, rv_snr_df, on='star_id', how='left')
table4 = pd.merge(table4, aat_df, on='star_id', how='left')
# print(np.array(table2.columns))

table4 = table4[['star_id', 'teff_y', 'randteff_std', 'logg_y', 'randlogg_std',
                 'Fe_it1', 'Fe_comberr', 'vmic']]

# Calculating the error in vmic (because I didn't do that oops)
def calc_vmic_error(teff, teff_err, logg, logg_err):
    logg_ = logg - 4.0
    teff_ = teff - 5500
    vmic_err = np.sqrt((3.16E-4 - 2.86E-4 * logg_)**2 * teff_err**2 + (-0.253 - 2.86E-4 * teff_ + 0.330 * logg_)**2 * logg_err**2)
    return vmic_err

table4['vmic_err'] = calc_vmic_error(table4['teff_y'], table4['randteff_std'], table4['logg_y'], table4['randlogg_std'])


table4 = table4.rename(columns={'teff_y': 'teff', 
                       'randteff_std': 'teff_err',
                       'logg_y': 'logg',
                       'randlogg_std': 'logg_err',
                       'Fe_it1': 'feh',
                       'Fe_comberr': 'feh_err',
                       'vmic': 'vmic',
                       'vmic_err': 'vmic_err'})
table4.sort_values(by='star_id', inplace=True)
display(table4)


# Storing into .fits file
t4_fits = Table.from_pandas(table4)
hdu4_fits = fits.BinTableHDU(t4_fits, name='STAR_PARAMS')
hdul4 = fits.HDUList([fits.PrimaryHDU(), hdu4_fits])

hdul4.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_4.fits', overwrite=True)
table4.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_4.csv', index=False)

,star_id,teff,teff_err,logg,logg_err,feh,feh_err,vmic,vmic_err
0,ra_0103-7050_s163,6298.806689,32.299219,4.492229,0.030431,-2.358308,0.060794,1.253413,0.011237
1,ra_0834-5220_s316,5959.304567,256.779041,4.286104,0.065557,-2.615424,0.137417,1.246679,0.063064
2,ra_1604-2712_s24,5002.684528,35.336965,1.891750,0.088924,-2.525669,0.060214,2.007753,0.078725
3,ra_1604-2712_s292,5783.249268,58.412127,3.404600,0.204987,-3.096067,0.066494,1.544869,0.112392
4,ra_1624-2150_s278,6381.865006,243.734656,3.973617,0.195541,-2.667178,0.058084,1.490113,0.127740
5,ra_1633-2814_s130,4997.869004,161.388411,1.849800,0.775219,-3.350497,0.068693,2.037393,0.652407
6,ra_1633-2814_s284,5261.142090,147.690554,2.447800,0.670789,-3.893281,0.064712,1.806730,0.480765
7,ra_1648-0653_s38,6039.830368,209.410462,3.646778,0.159170,-2.476114,0.059728,1.533073,0.120754
8,ra_1656-1433_s143,4791.462397,207.089148,1.406800,0.515657,-2.861975,0.060508,2.214264,0.516034
9,ra_1658-2454_s22,4852.253676,87.659488,2.164886,0.309023,-2.897512,0.076858,1.673293,0.220745


### Table 5 - Abundances
This table should have all of the abundances with their corresponding errors

In [30]:
# Loading in the stelalr parameters
abunds_path = '/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/'
abunds_files = [abunds_path + f for f in os.listdir(abunds_path) if f.endswith('I_abunds.csv')]
# abunds_files
abunds_df = pd.concat([pd.read_csv(f) for f in abunds_files], ignore_index=True)
abunds_df.keys()

table5 = abunds_df[['star_id', 'el', 'detection', 'bestfit_abund_xfe', 'bestfit_abund_comberr_xfe', 'upper_limit_xfe',
                    'bestfit_abund_xh', 'bestfit_abund_comberr_xh', 'upper_limit_xh']]
table5 = table5.rename(columns={'bestfit_abund_xfe': 'xfe',
                               'bestfit_abund_comberr_xfe': 'xfe_err',
                               'upper_limit_xfe': 'xfe_ul',
                               'bestfit_abund_xh': 'xh',
                               'bestfit_abund_comberr_xh': 'xh_err',
                               'upper_limit_xh': 'xh_ul'})
table5.sort_values(by='star_id', inplace=True)

# For those with upper limits, setting the error, xfe and xh values to NaN
for index, row in table5.iterrows():
    if row['detection'] == 'N':
        table5.at[index, 'xfe'] = np.nan
        table5.at[index, 'xfe_err'] = np.nan
        table5.at[index, 'xh'] = np.nan
        table5.at[index, 'xh_err'] = np.nan

display(table5)

# Storing into .fits file
t5_fits = Table.from_pandas(table5)
hdu5_fits = fits.BinTableHDU(t5_fits, name='ABUNDS')
hdul5 = fits.HDUList([fits.PrimaryHDU(), hdu5_fits])

hdul5.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_5.fits', overwrite=True)
table5.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_5.csv', index=False)

,star_id,el,detection,xfe,xfe_err,xfe_ul,xh,xh_err,xh_ul
0,ra_0103-7050_s163,Al_I,Y,-0.854418,0.032406,NaN,-3.213421,0.032406,NaN
205,ra_0103-7050_s163,Si_I,Y,0.314925,0.055803,NaN,-2.044078,0.055803,NaN
189,ra_0103-7050_s163,Sc_II,Y,-0.195520,0.041435,NaN,-2.554524,0.041435,NaN
173,ra_0103-7050_s163,Ni_I,N,NaN,NaN,0.012068,NaN,NaN,-2.346239
160,ra_0103-7050_s163,Na_I,Y,-0.367556,0.061959,NaN,-2.726559,0.061959,NaN
...,...,...,...,...,...,...,...,...,...
172,ra_1853-3255_s45,Na_I,Y,0.566409,0.068027,NaN,-2.372223,0.068027,NaN
188,ra_1853-3255_s45,Ni_I,Y,-0.012608,0.027940,NaN,-2.951240,0.027940,NaN
204,ra_1853-3255_s45,Sc_II,Y,0.065157,0.089305,NaN,-2.873475,0.089305,NaN
79,ra_1853-3255_s45,Co_I,Y,0.139834,0.047287,NaN,-2.798798,0.047287,NaN


### Table 6 - [C/N] abundances
This table should have the following:
1. star ID
2. [N/Fe]
3. [N/Fe] errors
4. [N/Fe] upper-limits (if required)
5. [C/N]
6. [C/N] errors
7. [C/N] lower-limits (if required)


In [31]:
# Loading in the C and N abundances
c_I_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/C_I_abunds.csv')
n_I_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/N_I_abunds.csv')


# Doing this for our stars
print("The following are N-enhanced:")
prodisk_list = []
halo_list = []
gse_list = []
all_list = []

for entry in c_I_abunds.iterrows():
    if entry[1]['detection'] == 'Y':
        star_id = entry[1]['star_id']
        nfe_entry = n_I_abunds[n_I_abunds['star_id'] == star_id]
        orb = entry[1]['orbit']
        if nfe_entry['detection'].values[0] == 'Y':
            
            # Getting raw [C/Fe] value by subtracting off the C correction
            cfe_vals = entry[1]['bestfit_abund_xfe']
            cfe_corr = entry[1]['cfe_corr']
            cfe_vals -= cfe_corr
            
            cfe_errs = entry[1]['bestfit_abund_comberr_xfe']
            nfe_vals = n_I_abunds[n_I_abunds['star_id'] == star_id]['bestfit_abund_xfe'].values[0]
            nfe_errs = n_I_abunds[n_I_abunds['star_id'] == star_id]['bestfit_abund_comberr_xfe'].values[0]
            feh_vals = entry[1]['feh']
            
            cn_vals = cfe_vals - nfe_vals
            cn_errs = np.sqrt(cfe_errs**2 + nfe_errs**2)
            
            if orb == 'pro_disk':
                prodisk_dict = {'star_id': star_id, 'detection': 'Y', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 
                                'cn': cn_vals, 'cn_err': cn_errs, 'feh': feh_vals}
                prodisk_list.append(prodisk_dict)
            elif orb == 'halo':
                halo_dict = {'star_id': star_id, 'detection': 'Y', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 
                                'cn': cn_vals, 'cn_err': cn_errs, 'feh': feh_vals}
                halo_list.append(halo_dict)
            elif orb == 'gse':
                gse_dict = {'star_id': star_id, 'detection': 'Y', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 
                                'cn': cn_vals, 'cn_err': cn_errs, 'feh': feh_vals}
                gse_list.append(gse_dict)
            
            all_list.append({'star_id': star_id, 'detection': 'Y', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 
                                'cn': cn_vals, 'cn_err': cn_errs, 'feh': feh_vals})
            
            if nfe_vals > 0.5 and cn_vals < -0.7:
                print(f"{star_id} ([N/Fe] = {nfe_vals:.2f} and [C/N] = {cn_vals:.2f})")

        if nfe_entry['detection'].values[0] == 'N':
            
            # Getting raw [C/Fe] value by subtracting off the C correction
            cfe_vals = entry[1]['bestfit_abund_xfe']
            cfe_corr = entry[1]['cfe_corr']
            cfe_vals -= cfe_corr
            
            cfe_errs = entry[1]['bestfit_abund_comberr_xfe']
            nfe_vals = n_I_abunds[n_I_abunds['star_id'] == star_id]['upper_limit_xfe'].values[0]
            nfe_errs = np.nan
            feh_vals = entry[1]['feh']
            
            cn_vals = cfe_vals - nfe_vals
            cn_errs = np.nan
            
            if orb == 'pro_disk':
                prodisk_dict = {'star_id': star_id, 'detection': 'N', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 'nfe_ul': nfe_vals,
                                'cn': cn_vals, 'cn_err': cn_errs, 'cn_ll': cn_vals, 'feh': feh_vals}
                prodisk_list.append(prodisk_dict)
            elif orb == 'halo':
                halo_dict = {'star_id': star_id, 'detection': 'N', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 'nfe_ul': nfe_vals,
                                'cn': cn_vals, 'cn_err': cn_errs, 'cn_ll': cn_vals, 'feh': feh_vals}
                halo_list.append(halo_dict)
            elif orb == 'gse':
                gse_dict = {'star_id': star_id, 'detection': 'N', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 'nfe_ul': nfe_vals,
                                'cn': cn_vals, 'cn_err': cn_errs, 'cn_ll': cn_vals, 'feh': feh_vals}
                gse_list.append(gse_dict)
            
            all_list.append({'star_id': star_id, 'detection': 'N', 'cfe': cfe_vals, 'cfe_err': cfe_errs, 'nfe': nfe_vals, 'nfe_err': nfe_errs, 'nfe_ul': nfe_vals,
                                'cn': cn_vals, 'cn_err': cn_errs, 'cn_ll': cn_vals, 'feh': feh_vals})

                
                
prodisk_df = pd.DataFrame(prodisk_list)
halo_df = pd.DataFrame(halo_list)
gse_df = pd.DataFrame(gse_list)
all_df = pd.DataFrame(all_list)

# Separating the non-detections from the detections
prodisk_detect_df = prodisk_df[prodisk_df['detection'] == 'Y']
gse_detect_df = gse_df[gse_df['detection'] == 'Y']
halo_detect_df = halo_df[halo_df['detection'] == 'Y']

prodisk_nondetect_df = prodisk_df[prodisk_df['detection'] == 'N']
gse_nondetect_df = gse_df[gse_df['detection'] == 'N']
halo_nondetect_df = halo_df[halo_df['detection'] == 'N']

The following are N-enhanced:
ra_1633-2814_s130 ([N/Fe] = 1.60 and [C/N] = -1.37)
ra_1656-1433_s143 ([N/Fe] = 1.28 and [C/N] = -1.32)


In [33]:
# Making table for paper
cn_tab = all_df[['star_id', 'detection', 'nfe', 'nfe_err', 'nfe_ul', 'cn', 'cn_err', 'cn_ll']]

# For those with non-detections, setting the error and cn values to NaN
for index, row in cn_tab.iterrows():
    if row['detection'] == 'N':
        cn_tab.at[index, 'cn'] = np.nan
        cn_tab.at[index, 'cn_err'] = np.nan
        cn_tab.at[index, 'nfe'] = np.nan
        cn_tab.at[index, 'nfe_err'] = np.nan

display(cn_tab)

# Storing into .fits file
t6_fits = Table.from_pandas(cn_tab)
hdu6_fits = fits.BinTableHDU(t6_fits, name='CN_ABUNDS')
hdul6 = fits.HDUList([fits.PrimaryHDU(), hdu6_fits])

hdul6.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_6.fits', overwrite=True)
cn_tab.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_6.csv', index=False)

,star_id,detection,nfe,nfe_err,nfe_ul,cn,cn_err,cn_ll
0,ra_0103-7050_s163,N,NaN,NaN,0.39,NaN,NaN,-0.021272
1,ra_0834-5220_s316,N,NaN,NaN,0.57,NaN,NaN,-0.348711
2,ra_1604-2712_s24,Y,0.296678,0.089472,NaN,-0.301184,0.120094,NaN
3,ra_1604-2712_s292,N,NaN,NaN,0.69,NaN,NaN,-0.645432
4,ra_1633-2814_s130,Y,1.601635,0.096741,NaN,-1.372407,0.127295,NaN
5,ra_1633-2814_s284,N,NaN,NaN,1.20,NaN,NaN,-0.485018
6,ra_1648-0653_s38,N,NaN,NaN,0.67,NaN,NaN,-0.261912
7,ra_1656-1433_s143,Y,1.282892,0.103416,NaN,-1.318919,0.243531,NaN
8,ra_1658-2454_s22,N,NaN,NaN,-1.11,NaN,NaN,0.780363
9,ra_1659-2154_s114,N,NaN,NaN,0.06,NaN,NaN,-0.056194


### Table 7 - $\alpha$ abundances
This table should have the following:
1. Star ID
2. [Fe/H]
3. [Fe/H] error
4. [alpha/Fe]
5. [alpha/Fe] error

In [94]:
# Loading in the Mg, Si, Ca, Ti abundances
mg_I_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Mg_I_abunds.csv')
si_I_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Si_I_abunds.csv')
ca_II_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Ca_II_abunds.csv')
ti_II_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Ti_II_abunds.csv')

# Doing this for our stars
prodisk_list = []
halo_list = []
gse_list = []
all_list = []

for entry in mg_I_abunds.iterrows():
    star_id = entry[1]['star_id']
    orb = entry[1]['orbit']
    si_I_entry = si_I_abunds[si_I_abunds['star_id'] == star_id]
    ca_II_entry = ca_II_abunds[ca_II_abunds['star_id'] == star_id]
    ti_II_entry = ti_II_abunds[ti_II_abunds['star_id'] == star_id]
    
    mg_vals = entry[1]['bestfit_abund_xfe']
    mg_errs = entry[1]['bestfit_abund_comberr_xfe']
    si_vals = si_I_entry['bestfit_abund_xfe'].values[0]
    si_errs = si_I_entry['bestfit_abund_comberr_xfe'].values[0]
    ca_vals = ca_II_entry['bestfit_abund_xfe'].values[0]
    ca_errs = ca_II_entry['bestfit_abund_comberr_xfe'].values[0]
    ti_vals = ti_II_entry['bestfit_abund_xfe'].values[0]
    ti_errs = ti_II_entry['bestfit_abund_comberr_xfe'].values[0]
    
    
    # alpha_vals = np.average([mg_vals, si_vals, ca_vals, ti_vals])
    # alpha_errs = 1/4 * np.sqrt(mg_errs**2 + si_errs**2 + ca_errs**2 + ti_errs**2)
    
    # Calculating weighted average
    alpha_arr = np.array([mg_vals, si_vals, ca_vals, ti_vals])
    alpha_weights = 1 / np.array([mg_errs, si_errs, ca_errs, ti_errs])**2
    
    # Weighted mean
    alpha_vals = np.sum(alpha_weights * alpha_arr) / np.sum(alpha_weights)

    # Uncertainty of weighted mean
    alpha_errs = np.sqrt(1 / np.sum(alpha_weights))
    
    feh_vals = entry[1]['feh']
    feh_errs = entry[1]['feh_err']
    
    all_dict = {'star_id': star_id, 'alpha': alpha_vals, 'alpha_err': alpha_errs,
                'feh': feh_vals, 'feh_err': feh_errs, 'mg': mg_vals, 'mg_err': mg_errs,
                'si': si_vals, 'si_err': si_errs, 'ca': ca_vals, 'ca_err': ca_errs, 'ti': ti_vals, 'ti_err': ti_errs}
    all_list.append(all_dict)
    
    if orb == 'pro_disk':
        prodisk_dict = {'star_id': star_id, 'alpha': alpha_vals, 'alpha_err': alpha_errs,
                        'feh': feh_vals, 'feh_err': feh_errs}
        prodisk_list.append(prodisk_dict)
    elif orb == 'halo':
        halo_dict = {'star_id': star_id, 'alpha': alpha_vals, 'alpha_err': alpha_errs,
                        'feh': feh_vals, 'feh_err': feh_errs}
        halo_list.append(halo_dict)
    elif orb == 'gse':
        gse_dict = {'star_id': star_id, 'alpha': alpha_vals, 'alpha_err': alpha_errs,
                        'feh': feh_vals, 'feh_err': feh_errs}
        gse_list.append(gse_dict)
    
    print(f"{star_id} ([α/Fe] = {alpha_vals:.2f} ± {alpha_errs:.2f})")
    # if eu_II_entry['detection'].values[0] == 'Y':
    #     ba_vals = entry[1]['bestfit_abund_xfe']
    #     eu_vals = eu_II_abunds[eu_II_abunds['star_id'] == star_id]['bestfit_abund_xfe'].values[0]
        
    #     baeu_vals = ba_vals - eu_vals
    #     if eu_vals >= 0.3 and eu_vals <= 1.0 and baeu_vals < 0:
    #         print(f"{star_id} ([Eu/Fe] = {eu_vals:.2f} and [Ba/Eu] = {baeu_vals:.2f})")

prodisk_df = pd.DataFrame(prodisk_list)
halo_df = pd.DataFrame(halo_list)
gse_df = pd.DataFrame(gse_list)
all_df = pd.DataFrame(all_list)

ra_0103-7050_s163 ([α/Fe] = 0.23 ± 0.02)
ra_0834-5220_s316 ([α/Fe] = 0.36 ± 0.02)
ra_1604-2712_s24 ([α/Fe] = 0.32 ± 0.02)
ra_1604-2712_s292 ([α/Fe] = 0.24 ± 0.02)
ra_1624-2150_s278 ([α/Fe] = 0.26 ± 0.01)
ra_1633-2814_s130 ([α/Fe] = 0.32 ± 0.02)
ra_1633-2814_s284 ([α/Fe] = 0.56 ± 0.02)
ra_1648-0653_s38 ([α/Fe] = 0.28 ± 0.02)
ra_1656-1433_s143 ([α/Fe] = 0.39 ± 0.02)
ra_1658-2454_s22 ([α/Fe] = 0.38 ± 0.02)
ra_1659-2154_s114 ([α/Fe] = 0.36 ± 0.02)
ra_1709-2130_s102 ([α/Fe] = 0.37 ± 0.02)
ra_1752-4300_s214 ([α/Fe] = 0.42 ± 0.02)
ra_1752-4300_s269 ([α/Fe] = 0.15 ± 0.02)
ra_1752-4300_s6 ([α/Fe] = 0.36 ± 0.02)
ra_1853-3255_s45 ([α/Fe] = 0.28 ± 0.02)


In [96]:
# Making table for paper
alpha_tab = all_df[['star_id', 'feh', 'feh_err', 'alpha', 'alpha_err']]
alpha_tab.rename(columns={'alpha': 'alphafe', 'alpha_err': 'alphafe_err'}, inplace=True)
display(alpha_tab)

# Storing into .fits file
t7_fits = Table.from_pandas(alpha_tab)
hdu7_fits = fits.BinTableHDU(t7_fits, name='ALPHA_ABUNDS')
hdul7 = fits.HDUList([fits.PrimaryHDU(), hdu7_fits])

hdul7.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_7.fits', overwrite=True)
alpha_tab.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_7.csv', index=False)

/tmp/ipykernel_4043784/1716424839.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  alpha_tab.rename(columns={'alpha': 'alphafe', 'alpha_err': 'alphafe_err'}, inplace=True)


,star_id,feh,feh_err,alphafe,alphafe_err
0,ra_0103-7050_s163,-2.359003,0.060794,0.227635,0.021678
1,ra_0834-5220_s316,-2.616194,0.137417,0.364046,0.019852
2,ra_1604-2712_s24,-2.530153,0.060214,0.323007,0.017465
3,ra_1604-2712_s292,-3.095019,0.066494,0.243211,0.015925
4,ra_1624-2150_s278,-2.667603,0.058084,0.255039,0.011936
5,ra_1633-2814_s130,-3.349203,0.068693,0.318427,0.018202
6,ra_1633-2814_s284,-3.893474,0.064712,0.555307,0.017111
7,ra_1648-0653_s38,-2.475968,0.059728,0.284566,0.016477
8,ra_1656-1433_s143,-2.858919,0.060508,0.386516,0.016409
9,ra_1658-2454_s22,-2.897767,0.076858,0.376241,0.020744


### Table 8 - [Sr/Ba] abundances
This table should have the following:
1. Star ID
2. [Ba/H]
3. [Ba/H] error
4. [Ba/H] upper-limit (if required)
5. [Sr/Ba]
6. [Sr/Ba] error
7. [Sr/Ba] lower-limit (if required)

In [41]:
# Loading in the Sr, Ba, Eu abundances
sr_II_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Sr_II_abunds.csv')
ba_II_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Ba_II_abunds.csv')
eu_II_abunds = pd.read_csv('/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/Eu_II_abunds.csv')

prodisk_list = []
halo_list = []
gse_list = []
all_list = []

for entry in sr_II_abunds.iterrows():
    if entry[1]['detection'] == 'Y':
        star_id = entry[1]['star_id']
        bafe_entry = ba_II_abunds[ba_II_abunds['star_id'] == star_id]
        orb = entry[1]['orbit']
        if bafe_entry['detection'].values[0] == 'Y':
            sr_vals = entry[1]['bestfit_abund_xfe']
            sr_errs = entry[1]['bestfit_abund_comberr_xfe']
            ba_vals = ba_II_abunds[ba_II_abunds['star_id'] == star_id]['bestfit_abund_xfe'].values[0]
            ba_xh_vals = ba_II_abunds[ba_II_abunds['star_id'] == star_id]['bestfit_abund_xh'].values[0]
            ba_errs = ba_II_abunds[ba_II_abunds['star_id'] == star_id]['bestfit_abund_comberr_xfe'].values[0]
            ba_xh_errs = ba_II_abunds[ba_II_abunds['star_id'] == star_id]['bestfit_abund_comberr_xh'].values[0]
            feh_vals = entry[1]['feh']
            feh_errs = entry[1]['feh_err']
            
            srba_vals = sr_vals - ba_vals
            srba_errs = np.sqrt(sr_errs**2 + ba_errs**2)
            
            eufe_entry = eu_II_abunds[eu_II_abunds['star_id'] == star_id]
            if bafe_entry['detection'].values[0] == 'Y':
                eu_vals = eu_II_abunds[eu_II_abunds['star_id'] == star_id]['bestfit_abund_xfe'].values[0]
                eu_errs = eu_II_abunds[eu_II_abunds['star_id'] == star_id]['bestfit_abund_comberr_xfe'].values[0]
                baeu_vals = ba_vals - eu_vals
                baeu_errs = np.sqrt(sr_errs**2 + eu_errs**2)
                
                print(star_id)
                print(baeu_vals, baeu_errs)
            
            all_dict = {'star_id': star_id, 'detection': 'Y', 'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'srba': srba_vals, 'srba_err': srba_errs, 'feh': feh_vals, 'feh_err': feh_errs,
                                'bah': ba_xh_vals, 'bah_err': ba_xh_errs}
            all_list.append(all_dict)
            
            if orb == 'pro_disk':
                prodisk_dict = {'star_id': star_id, 'detection': 'Y',  'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'srba': srba_vals, 'srba_err': srba_errs, 'feh': feh_vals, 'feh_err': feh_errs,
                                'bah': ba_xh_vals, 'bah_err': ba_xh_errs}
                prodisk_list.append(prodisk_dict)
            elif orb == 'halo':
                halo_dict = {'star_id': star_id, 'detection': 'Y',  'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'srba': srba_vals, 'srba_err': srba_errs, 'feh': feh_vals, 'feh_err': feh_errs,
                                'bah': ba_xh_vals, 'bah_err': ba_xh_errs}
                halo_list.append(halo_dict)
            elif orb == 'gse':
                gse_dict = {'star_id': star_id, 'detection': 'Y',  'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'srba': srba_vals, 'srba_err': srba_errs, 'feh': feh_vals, 'feh_err': feh_errs,
                                'bah': ba_xh_vals, 'bah_err': ba_xh_errs}
                gse_list.append(gse_dict)      
                
        elif bafe_entry['detection'].values[0] == 'N':
            sr_vals = entry[1]['bestfit_abund_xfe']
            sr_errs = entry[1]['bestfit_abund_comberr_xfe']
            ba_vals = ba_II_abunds[ba_II_abunds['star_id'] == star_id]['upper_limit_xfe'].values[0]
            ba_xh_vals = ba_II_abunds[ba_II_abunds['star_id'] == star_id]['upper_limit_xh'].values[0]
            ba_errs = np.nan
            ba_xh_errs = np.nan
            feh_vals = entry[1]['feh']
            feh_errs = entry[1]['feh_err']
            
            srba_vals = sr_vals - ba_vals
            srba_errs = np.nan
            
            eufe_entry = eu_II_abunds[eu_II_abunds['star_id'] == star_id]
            
            all_dict = {'star_id': star_id, 'detection': 'N', 'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'bafe_ul': ba_vals, 'srba': srba_vals, 'srba_err': srba_errs, 'srba_ll': srba_vals, 
                                'feh': feh_vals, 'feh_err': feh_errs, 'bah': ba_xh_vals, 'bah_err': ba_xh_errs, 'bah_ul': ba_xh_vals}
            all_list.append(all_dict)
            
            if orb == 'pro_disk':
                prodisk_dict = {'star_id': star_id, 'detection': 'N', 'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'bafe_ul': ba_vals, 'srba': srba_vals, 'srba_err': srba_errs, 'srba_ll': srba_vals, 
                                'feh': feh_vals, 'feh_err': feh_errs, 'bah': ba_xh_vals, 'bah_err': ba_xh_errs, 'bah_ul': ba_xh_vals}
                prodisk_list.append(prodisk_dict)
            elif orb == 'halo':
                halo_dict = {'star_id': star_id, 'detection': 'N', 'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'bafe_ul': ba_vals, 'srba': srba_vals, 'srba_err': srba_errs, 'srba_ll': srba_vals, 
                                'feh': feh_vals, 'feh_err': feh_errs, 'bah': ba_xh_vals, 'bah_err': ba_xh_errs, 'bah_ul': ba_xh_vals}
                halo_list.append(halo_dict)
            elif orb == 'gse':
                gse_dict = {'star_id': star_id, 'detection': 'N', 'srfe': sr_vals, 'bafe': ba_vals, 'srfe_err': sr_errs,
                                'bafe_err': ba_errs, 'bafe_ul': ba_vals, 'srba': srba_vals, 'srba_err': srba_errs, 'srba_ll': srba_vals, 
                                'feh': feh_vals, 'feh_err': feh_errs, 'bah': ba_xh_vals, 'bah_err': ba_xh_errs, 'bah_ul': ba_xh_vals}
                gse_list.append(gse_dict)      

                
prodisk_df = pd.DataFrame(prodisk_list)
halo_df = pd.DataFrame(halo_list)
gse_df = pd.DataFrame(gse_list)
all_df = pd.DataFrame(all_list)

# Separating the non-detections from the detections
prodisk_detect_df = prodisk_df[prodisk_df['detection'] == 'Y']
gse_detect_df = gse_df[gse_df['detection'] == 'Y']
halo_detect_df = halo_df[halo_df['detection'] == 'Y']

prodisk_nondetect_df = prodisk_df[prodisk_df['detection'] == 'N']
gse_nondetect_df = gse_df[gse_df['detection'] == 'N']
halo_nondetect_df = halo_df[halo_df['detection'] == 'N']


ra_0103-7050_s163
-0.9613192987092622 0.1113933869806802
ra_0834-5220_s316
2.3130415607279753 0.07615082674251411
ra_1604-2712_s24
2.912133610370418 0.8739424325485757
ra_1648-0653_s38
-1.6060358780583242 1.5057469366824092
ra_1656-1433_s143
-0.4904084019053243 0.04800493658272009
ra_1658-2454_s22
2.2299796700447434 0.10845616703326573
ra_1659-2154_s114
-1.7817399209652376 0.058322658802958546
ra_1709-2130_s102
-1.5342206645138665 1.2136129810353575
ra_1752-4300_s214
-0.5661414056436436 0.5765477919423332
ra_1752-4300_s269
1.940583183925319 1.7795155009240404
ra_1752-4300_s6
-0.9790399256330431 0.15101981049018537
ra_1853-3255_s45
-0.184052607230616 0.08236417102096437


In [42]:
# Making table for paper
srba_tab = all_df[['star_id', 'detection', 'bah', 'bah_err', 'bah_ul', 'srba', 'srba_err', 'srba_ll']]

# For those with non-detections, setting the error and cn values to NaN
for index, row in srba_tab.iterrows():
    if row['detection'] == 'N':
        srba_tab.at[index, 'bah'] = np.nan
        srba_tab.at[index, 'bah_err'] = np.nan
        srba_tab.at[index, 'srba'] = np.nan
        srba_tab.at[index, 'srba_err'] = np.nan

display(srba_tab)

# Storing into .fits file
t8_fits = Table.from_pandas(srba_tab)
hdu8_fits = fits.BinTableHDU(t6_fits, name='SRBA_ABUNDS')
hdul8 = fits.HDUList([fits.PrimaryHDU(), hdu6_fits])

hdul8.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_8.fits', overwrite=True)
srba_tab.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_8.csv', index=False)




,star_id,detection,bah,bah_err,bah_ul,srba,srba_err,srba_ll
0,ra_0103-7050_s163,Y,-2.673533,0.060650,NaN,0.291698,0.089345,NaN
1,ra_0834-5220_s316,Y,-2.989086,0.118631,NaN,0.477883,0.135505,NaN
2,ra_1604-2712_s24,Y,-3.366295,0.042585,NaN,0.555209,0.103829,NaN
3,ra_1604-2712_s292,N,NaN,NaN,-3.430750,NaN,NaN,-0.900823
4,ra_1624-2150_s278,N,NaN,NaN,-3.045237,NaN,NaN,0.045758
5,ra_1633-2814_s130,N,NaN,NaN,-4.770016,NaN,NaN,0.165229
6,ra_1633-2814_s284,N,NaN,NaN,-4.436362,NaN,NaN,1.036459
7,ra_1648-0653_s38,Y,-3.413303,0.123030,NaN,0.954732,0.148508,NaN
8,ra_1656-1433_s143,Y,-2.027039,0.075456,NaN,0.386473,0.082875,NaN
9,ra_1658-2454_s22,Y,-4.222635,0.026682,NaN,0.798108,0.099801,NaN


# Supplementary tables

### Table S1 - Line list windows
This table should have all of the windows used in Korg


In [99]:
# Getting the windows for each of the elements (from Table3 above)
els_list = ['C_I', 'N_I', 'Mg_I', 'Al_I', 'Si_I', 'Ca_II', 'Sc_II',
             'Ti_II', 'Cr_I', 'Mn_I', 'Fe_I', 'Co_I', 'Ni_I', 'Sr_II', 'Ba_II', 'Eu_II']

els_path = '/data/misfit/benlowe/xshooter/p113/linelist/abund_lines/v3'
els_files = [els_path + '/' + f + '/windows_' + f + '.csv' for f in os.listdir(els_path) if f in els_list]

windows_df = pd.concat([pd.read_csv(f) for f in els_files], ignore_index=True)
# display(windows_df)

# Fixing up species names
windows_df['species'] = windows_df['species'].replace({'Al 1': 'Al_I',
                                                 'Ba 2': 'Ba_II',
                                                 'Ca 2': 'Ca_II',
                                                 'Co 1': 'Co_I',
                                                 'Cr 1': 'Cr_I',
                                                 'Eu 2': 'Eu_II',
                                                 'Fe 1': 'Fe_I',
                                                 'Mg 1': 'Mg_I',
                                                 'Mn 1': 'Mn_I',
                                                 'Ni 1': 'Ni_I',
                                                 'Sc 2': 'Sc_II',
                                                 'Si 1': 'Si_I',
                                                 'Sr 2': 'Sr_II',
                                                 'Ti 2': 'Ti_II',
                                                 'C 1': 'CH',
                                                 'N 1': 'NH'})

# Rearanging the order so its ordered based by atomic number
el_order = ['CH', 'NH', 'Na_I', 'Mg_I', 'Al_I', 'Si_I', 'Ca_II', 'Sc_II',
             'Ti_II', 'Cr_I', 'Mn_I', 'Fe_I', 'Co_I', 'Ni_I', 'Sr_II', 'Ba_II', 'Eu_II']
el_type = CategoricalDtype(categories=el_order, ordered=True)

windows_df['species'] = windows_df['species'].astype(el_type)
windows_df.sort_values(by='species', inplace=True)

windows_df.rename(columns={'wavetop': 'line',
                          'wavebase': 'wavemin',
                          'wavepeak': 'wavemax',
                          'species': 'el'}, inplace=True)

windows_df = windows_df[['el', 'line', 'wavemin', 'wavemax']]

display(windows_df)

# Storing into .fits file
ts3_fits = Table.from_pandas(windows_df)
hdus3_fits = fits.BinTableHDU(ts3_fits, name='ABUNDS_WINDOWS')
hduls3 = fits.HDUList([fits.PrimaryHDU(), hdus3_fits])


hduls3.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_S1.fits', overwrite=True)
windows_df.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_S1.csv', index=False)

,el,line,wavemin,wavemax
32,CH,4330.000,4260.000,4300.313
33,NH,3365.500,3355.000,3360.220
23,Mg_I,8811.757,8801.757,8806.757
22,Mg_I,5187.500,5179.000,5183.604
21,Mg_I,5177.500,5162.500,5167.322
0,Al_I,3962.500,3957.500,3961.520
27,Si_I,3909.000,3901.200,3905.523
6,Ca_II,8669.000,8655.000,8662.138
5,Ca_II,8549.000,8535.000,8542.198
4,Ca_II,8504.020,8491.020,8498.020


### Table S2 - Na windows
This table should have all of the windows for Na used in Korg


In [100]:

na_window_path = '/data/misfit/benlowe/xshooter/p113/linelist/abund_lines/v3/Na_I/'
na_window_files = [na_window_path + f for f in os.listdir(na_window_path) if f.startswith('ra_') and f.endswith('_windows_Na_I.csv')]

# Getting the dataframes for each star and combining
all_dfs = []
for f in na_window_files:
    
    star_id = 'ra_' + f.split('/')[-1].split('_')[1] + '_' + f.split('/')[-1].split('_')[2]
    
    # Opening up the na windows file
    na_df = pd.read_csv(f)
    na_df['star_id'] = star_id
    
    na_df = na_df[['star_id', 'wavepeak', 'wavebase', 'wavetop']]
    all_dfs.append(na_df)
na_windows_df = pd.concat(all_dfs, ignore_index=True)
    
# Renaming columns and sorting
na_windows_df.rename(columns={'wavepeak': 'line',
                              'wavebase': 'wavemin',
                              'wavetop': 'wavemax'}, inplace=True)
na_windows_df.sort_values(by=['star_id', 'line'], inplace=True)

# Removing stars with interstellar Na contamination
contaminated_stars = ['ra_1604-2712_s292', 'ra_1648-0653_s38', 'ra_1752-4300_s269']
na_windows_df = na_windows_df[~na_windows_df['star_id'].isin(contaminated_stars)]
display(na_windows_df)

# Storing into .fits file
ts4_fits = Table.from_pandas(na_windows_df)
hdus4_fits = fits.BinTableHDU(ts4_fits, name='NA_WINDOWS')
hduls4 = fits.HDUList([fits.PrimaryHDU(), hdus4_fits])


hduls4.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_S2.fits', overwrite=True)
na_windows_df.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_S2.csv', index=False)

,star_id,line,wavemin,wavemax
0,ra_0103-7050_s163,5889.951,5885.000,5894.0
1,ra_0103-7050_s163,5895.924,5894.500,5900.0
2,ra_0834-5220_s316,5889.951,5888.800,5892.5
3,ra_0834-5220_s316,5895.924,5894.424,5900.0
4,ra_1604-2712_s24,5889.951,5887.500,5891.0
5,ra_1604-2712_s24,5895.924,5893.500,5900.0
8,ra_1624-2150_s278,5889.951,5887.000,5890.8
9,ra_1624-2150_s278,5895.924,5893.000,5900.0
10,ra_1633-2814_s130,5889.951,5887.500,5891.0
11,ra_1633-2814_s130,5895.924,5894.000,5900.0


### Table S3 - NLTE corrections
This table should have the following:
1. Star ID
2. CaII line
3. Continuum offset applied
4. NLTE correction
5. Corrected [Ca/Fe] abundance

Saving the NLTE corrections to Ca II

In [35]:
# The corrections are saved in each star directory, so need to go into each to grab the corrections
abunds_path = '/data/misfit/benlowe/xshooter/p113/korg_data/abunds/final_iter9/'
caII_dirs = [abunds_path + d + '/Ca_II/iter_1/Ca_results.csv' for d in os.listdir(abunds_path) if os.path.isdir(os.path.join(abunds_path, d))]

# Running through each Ca II df, extracting the corrections and contiuum offset
caII_nlte_list = []
for star in caII_dirs:
    star_df = pd.read_csv(star)
    
    star_id = [star.split('/')[-4]] * len(star_df)
    line = [float(i.split('_')[1]) for i in star_df['line'].to_numpy()]
    cntm_offset = [float(j) for j in star_df['cntm_offset'].to_numpy()]
    nlte_corr = [float(k) for k in star_df['nlte_corr'].to_numpy()]
    xfe_abund = [float(l) for l in star_df['abund_xfe'].to_numpy()]
    
    star_caII_df = pd.DataFrame({
        'star_id': star_id,
        'line': line,
        'cntm_offset': cntm_offset,
        'nlte_corr': nlte_corr,
        'cafe': xfe_abund
    })
        
    caII_nlte_list.append(star_caII_df)

# Creating a DataFrame from the list of dictionaries
caII_nlte_df = pd.concat(caII_nlte_list, ignore_index=True)
caII_nlte_df.sort_values(by='star_id', inplace=True)
display(caII_nlte_df)

# Storing into .fits file
tS1_fits = Table.from_pandas(caII_nlte_df)
hduS1_fits = fits.BinTableHDU(tS1_fits, name='CAII_NLTE')
hdulS1 = fits.HDUList([fits.PrimaryHDU(), hduS1_fits])

hdulS1.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_S1.fits', overwrite=True)
caII_nlte_df.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_S1.csv', index=False)

,star_id,line,cntm_offset,nlte_corr,cafe
17,ra_0103-7050_s163,8662.138,-0.015,-0.370259,0.231460
16,ra_0103-7050_s163,8542.198,-0.010,-0.341881,0.157173
15,ra_0103-7050_s163,8498.020,-0.010,-0.328171,0.299693
20,ra_0834-5220_s316,8662.138,-0.015,-0.343445,0.327469
19,ra_0834-5220_s316,8542.198,-0.005,-0.323131,0.315511
18,ra_0834-5220_s316,8498.020,-0.010,-0.322462,0.501991
14,ra_1604-2712_s24,8662.138,-0.000,-0.289597,0.300253
13,ra_1604-2712_s24,8542.198,-0.000,-0.287465,0.281513
12,ra_1604-2712_s24,8498.020,-0.010,-0.306686,0.322410
23,ra_1604-2712_s292,8662.138,-0.015,-0.412610,0.449234


### Table S4 - Evolutionary corrections
This table should have the following:
1. Star ID
2. [C/Fe] evolutionary correction

Saving the evolutionary corrections

In [25]:
# Loading in the evolutionary corrections
c_corr_df = pd.read_csv('/data/misfit/benlowe/xshooter/p113/C_evolve_corrections/cfe_corrections.csv')
c_corr_df = c_corr_df[['star_id', 'cfe_corr']]

# Storing into .fits file
tS2_fits = Table.from_pandas(c_corr_df)
hduS2_fits = fits.BinTableHDU(tS2_fits, name='C_EVO_CORR')
hdulS2 = fits.HDUList([fits.PrimaryHDU(), hduS2_fits])

hdulS2.writeto('/data/misfit/benlowe/xshooter/p113/data_tables/table_S2.fits', overwrite=True)
c_corr_df.to_csv('/data/misfit/benlowe/xshooter/p113/data_tables/table_S2.csv', index=False)

c_corr_df

,star_id,cfe_corr
0,ra_0103-7050_s163,0.00
1,ra_0834-5220_s316,0.00
2,ra_1604-2712_s24,0.12
3,ra_1604-2712_s292,0.00
4,ra_1624-2150_s278,0.00
5,ra_1633-2814_s130,0.03
6,ra_1633-2814_s284,0.00
7,ra_1648-0653_s38,0.00
8,ra_1656-1433_s143,0.46
9,ra_1658-2454_s22,0.00
